In [ ]:
import numpy as np
import networkx as nx
from sklearn.neighbors import NearestNeighbors


def knn_graph(points, k, mutual=False):

    n = len(points)

    nbrs = NearestNeighbors(n_neighbors=k+1)
    nbrs.fit(points)

    _, indices = nbrs.kneighbors(points)

    G = nx.Graph()
    G.add_nodes_from(range(n))
    if mutual:

        neighbor_sets = [set(row[1:]) for row in indices]

        for i in range(n):
            for j in indices[i][1:]:
                if i in neighbor_sets[j]:
                    G.add_edge(i,j)

    else:

        for i in range(n):
            for j in indices[i][1:]:
                G.add_edge(i,j)

    return G


from collections import defaultdict
def build_edges_triangles(G):
    # -------------------------
    # edges
    # -------------------------

    edges = []
    edge_id = {}

    for i,j in G.edges():

        if i>j:
            i,j=j,i

        edge_id[(i,j)] = len(edges)
        edges.append((i,j))

    # -------------------------
    # adjacency
    # -------------------------

    neighbors = {
        v:set(G.neighbors(v))
        for v in G.nodes()
    }

    # -------------------------
    # triangles
    # -------------------------

    triangles = []

    for i in G.nodes():

        Ni = neighbors[i]

        for j in Ni:

            if j<=i:
                continue

            common = Ni & neighbors[j]

            for k in common:

                if k<=j:
                    continue

                triangles.append((i,j,k))

    return edges, edge_id, triangles


from scipy.sparse import csr_matrix
def build_B1(n_vertices, edges):

    rows=[]
    cols=[]
    vals=[]

    for e,(i,j) in enumerate(edges):

        rows.extend([i,j])
        cols.extend([e,e])
        vals.extend([-1,1])

    return csr_matrix(
        (vals,(rows,cols)),
        shape=(n_vertices,len(edges))
    )
def build_B2(edges, edge_id, triangles):
    rows=[]
    cols=[]
    vals=[]

    for t,(i,j,k) in enumerate(triangles):

        e0=edge_id[(j,k)]
        e1=edge_id[(i,k)]
        e2=edge_id[(i,j)]

        rows.extend([e0,e1,e2])
        cols.extend([t,t,t])

        vals.extend([1,-1,1])

    return csr_matrix(
        (vals,(rows,cols)),
        shape=(len(edges),len(triangles))
    )
def edge_flow_from_vector_field(edges, points, vectors):
    omega=np.zeros(len(edges))

    for m,(i,j) in enumerate(edges):

        e=points[j]-points[i]

        L=np.linalg.norm(e)

        if L==0:
            continue

        t=e/L

        v=0.5*(vectors[i]+vectors[j])

        omega[m]=np.dot(v,t)

    return omega
from scipy.sparse.linalg import lsmr
def hodge_decomposition(B1, B2, omega):

    # vertex potential
    alpha = lsmr(B1.T, omega)[0]

    exact = B1.T @ alpha

    x = lsmr(B2, omega)[0]
    coexact = B2 @ x

    harmonic = omega - exact - coexact

    return alpha, exact, coexact, harmonic


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from torchvision import transforms
from torchvision.datasets import STL10
from torch.utils.data import DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from timm.models.vision_transformer import VisionTransformer

weight_root_path="PATH"

class SimCLR(nn.Module):

    def __init__(self):

        super().__init__()

        self.encoder = VisionTransformer(
            img_size=224,
            patch_size=16,
            in_chans=3,
            num_classes=0,      
            embed_dim=256,       
            depth=4,             
            num_heads=4,         
            mlp_ratio=4.0,
            qkv_bias=True,
        )

        feature_dim = self.encoder.num_features

        self.projector = nn.Sequential(
            nn.Linear(feature_dim,2048),
            nn.ReLU(inplace=True),
            nn.Linear(2048,128)
        )

    def forward(self,x):

        h = self.encoder(x)
        z = self.projector(h)

        z = F.normalize(z,dim=1)

        return h,z
import torch
from torch.utils.data import DataLoader
from torchvision.datasets import STL10
from torchvision import transforms
import numpy as np


# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ----------------------
# STL10 labeled dataset
# ----------------------

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

dataset = STL10(
    root="./data",
    split="train",   # labeled train data
    download=True,
    transform=transform
)

loader = DataLoader(
    dataset,
    batch_size=128,
    shuffle=False,
    num_workers=4
)


In [ ]:
# ----------------------
# SimCLR model load
# ----------------------
full_feature_list=[]
full_projs_list=[]
model = SimCLR()
weightpaths=["PATHS FOR WEIGHTS HERE"
             ]
for weightpath in weightpaths:
    print(weightpath)
    checkpoint = torch.load(
        weightpath,
        map_location=device
    )
    print("checkpoint loaded")
    model.load_state_dict(checkpoint)

    model = model.to(device)
    model.eval()


    # ----------------------
    # Feature extraction
    # ----------------------

    tokens_list = []
    labels_list = []
    proj_list = []

    with torch.no_grad():

        for images, targets in loader:
            images = images.to(device)
            h,z = model(images)
            tokens_list.append(h.cpu())
            proj_list.append(z.cpu())
            labels_list.append(targets)

    tokens = torch.cat(tokens_list, dim=0)   
    projs = torch.cat(proj_list,dim=0)
    labels = torch.cat(labels_list, dim=0)
    print("token calculated")

    tokens = tokens.numpy()
    projs = projs.numpy()
    labels = labels.numpy()
    full_feature_list.append(tokens)
    full_projs_list.append(projs)

In [ ]:
full_feature_arr=np.array(full_feature_list)
print(full_feature_arr.shape)
move_vector=[full_feature_arr[i+1]-full_feature_arr[i] for i in range (0,len(full_feature_arr)-1)]
move_vector=np.array(move_vector)
move_vector_relative=move_vector-np.mean(move_vector,axis=1,keepdims=True)

In [ ]:
def process_one(index):

    points=full_feature_arr[index]
    vectors=move_vector_relative[index]
    #################################################
    # 1. kNN graph
    #################################################
    G = knn_graph(points, k=100)

    #################################################
    # 2. edge・triangle
    #################################################
    edges, edge_id, triangles = build_edges_triangles(G)

    #################################################
    # 3. Boundary matrices
    #################################################
    B1 = build_B1(len(points), edges)

    B2 = build_B2(edges, edge_id, triangles)

    #################################################
    # 4. Edge flow
    #################################################
    omega = edge_flow_from_vector_field(
        edges,
        points,
        vectors,
    )

    #################################################
    # 5. Hodge decomposition
    #################################################
    alpha, exact, coexact, harmonic = hodge_decomposition(
        B1,
        B2,
        omega,
    )

    L=B1@B1.T

    lam=0.05### 
    
    alpha_smooth = spsolve(
        identity(L.shape[0]) + lam*L,
        alpha
    )
    return alpha_smooth,alpha
from joblib import Parallel, delayed
from tqdm.auto import tqdm

indices = range(len(full_feature_arr)-1)

n_workers = 20

print("Starting parallel processing...")

results = Parallel(
    n_jobs=n_workers,
    backend="loky",      
    verbose=10           
)(
    delayed(process_one)(idx)
    for idx in tqdm(indices, desc="Submitting jobs")
)
smooth_potentials,potentials=zip(*results)

print("Finished")
smooth_potentials=np.array(smooth_potentials)
potentials=np.array(potentials)
print(smooth_potentials.shape)
### Modify norm
smooth_potentials_norm=smooth_potentials/np.linalg.norm(smooth_potentials,axis=1,keepdims=True)
from sklearn.preprocessing import StandardScaler
smooth_potentials_scaled = StandardScaler().fit_transform(smooth_potentials_norm)
from sklearn.decomposition import PCA
pca = PCA()
embedding = pca.fit_transform(smooth_potentials_scaled)
import umap
reducer=umap.UMAP(random_state=3)
embedding2=reducer.fit_transform(embedding)
num_labels=np.arange(len(embedding2))
plt.scatter(embedding2[:,0],embedding2[:,1],c=(np.arange(len(embedding2))),cmap="cool")
for i in range(len(embedding2)):
    plt.text(embedding2[i,0],embedding2[i,1], num_labels[i]+1)
plt.plot(embedding2[:,0],embedding2[:,1], color='gray', alpha=0.3, linewidth=1)
plt.show()